# 🧬 Segmentasi & Klasifikasi Sel Darah Putih (WBC)
## Menggunakan U-Net dan CNN
---
**Topik 18** — Informatika Biomedik (Biomedical Image Processing)

Notebook ini berisi pipeline lengkap:
1. 📂 Memeriksa dataset (BCCD untuk segmentasi, Blood Cell Images untuk klasifikasi)
2. 🧩 Membangun & melatih **U-Net** (segmentasi)
3. 🏷️ Membangun & melatih **CNN/ResNet50** (klasifikasi)
4. 📊 Evaluasi: Dice, IoU, akurasi, precision/recall/F1, confusion matrix

> Catatan: Jalankan sel dari atas ke bawah. Pastikan dataset sudah disiapkan dengan `siapkan_dataset.py`.

## 0. Instal Dependensi
Jalankan sel berikut hanya bila perlu (misal di lingkungan baru).
```bash
pip install tensorflow opencv-python numpy matplotlib scikit-learn
```

In [ ]:
# ============================================================
# 0. IMPOR & PENGATURAN
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("OpenCV version:", cv2.__version__)

In [ ]:
# ============================================================
# 0b. KONSTANTA
# ============================================================
IMG_SIZE = 256        # ukuran input
BATCH_SIZE = 8
EPOCHS_SEG = 50       # epoch segmentasi
EPOCHS_CLS = 30       # epoch klasifikasi
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

# Path dataset (sesuaikan bila perlu)
SEG_IMG_DIR = "dataset/bccd/train_images"
SEG_MSK_DIR = "dataset/bccd/train_masks"
CLS_DATA_DIR = "dataset/blood_cells"

print("Konstanta siap.")

## 1. Pemeriksaan Dataset

In [ ]:
# ============================================================
# 1a. CEK STRUKTUR DATASET
# ============================================================
for label, path in [("Segmentasi - Image", SEG_IMG_DIR),
                    ("Segmentasi - Mask", SEG_MSK_DIR),
                    ("Klasifikasi", CLS_DATA_DIR)]:
    if os.path.isdir(path):
        items = os.listdir(path)
        print(f"✓ {label}: {path} -> {len(items)} entri")
        if os.path.isdir(os.path.join(path, items[0])):
            for sub in items[:8]:
                n = len(os.listdir(os.path.join(path, sub)))
                print(f"    - {sub}: {n} gambar")
    else:
        print(f"✗ {label}: {path} -> TIDAK ditemukan!")

In [ ]:
# ============================================================
# 1b. LOADING DATA SEGMENTASI (BCCD)
# ============================================================
def load_bccd(img_dir, msk_dir):
    """Membaca gambar + mask BCCD (matching via stem nama file)."""
    imgs, masks = [], []
    mask_index = {}
    for m in os.listdir(msk_dir):
        if m.lower().endswith((".jpg", ".png", ".jpeg")):
            mask_index[os.path.splitext(m)[0]] = os.path.join(msk_dir, m)
    files = sorted([f for f in os.listdir(img_dir)
                    if f.lower().endswith((".jpg", ".png", ".jpeg"))])
    for fname in files:
        # Gambar
        img = cv2.imread(os.path.join(img_dir, fname))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img.astype("float32") / 255.0

        # Mask via stem (tanpa ekstensi)
        msk_path = mask_index.get(os.path.splitext(fname)[0])
        if msk_path is None:
            continue
        msk = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        if msk is None:
            continue
        msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE))
        msk = (msk > 0).astype("float32")
        msk = np.expand_dims(msk, axis=-1)

        imgs.append(img)
        masks.append(msk)

    return np.array(imgs), np.array(masks)

if os.path.isdir(SEG_IMG_DIR) and os.path.isdir(SEG_MSK_DIR):
    imgs, masks = load_bccd(SEG_IMG_DIR, SEG_MSK_DIR)
    print(f"Jumlah sample segmentasi: {len(imgs)}")
    print(f"Bentuk gambar: {imgs.shape}, Bentuk mask: {masks.shape}")
else:
    imgs = masks = None
    print("Dataset segmentasi belum ada.")

In [ ]:
# ============================================================
# 1c. VISUALISASI SAMPEL SEGMENTASI
# ============================================================
if imgs is not None and len(imgs) > 0:
    n = min(4, len(imgs))
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    for i in range(n):
        axes[0, i].imshow(imgs[i])
        axes[0, i].set_title(f"Image {i}"); axes[0, i].axis("off")
        axes[1, i].imshow(masks[i, :, :, 0], cmap="gray")
        axes[1, i].set_title(f"Mask {i}"); axes[1, i].axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Tidak ada data ditampilkan.")

## 2. Metrik Evaluasi untuk Segmentasi

In [ ]:
# ============================================================
# 2. METRIK KUSTOM
# ============================================================
def iou_metric(y_true, y_pred):
    """Intersection over Union (Jaccard)."""
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    y_true = tf.cast(y_true, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
    return (intersection + 1e-7) / (union + 1e-7)

def dice_metric(y_true, y_pred):
    """Sorensen-Dice coefficient."""
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    y_true = tf.cast(y_true, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2.0 * intersection + 1e-7) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + 1e-7)

print("Metrik siap.")

## 3. Arsitektur U-Net

In [ ]:
# ============================================================
# 3. U-NET
# ============================================================
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu',
                      kernel_initializer='he_normal')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu',
                      kernel_initializer='he_normal')(x)
    return x

def build_unet(input_size=(IMG_SIZE, IMG_SIZE, 3)):
    inputs = layers.Input(input_size)

    # ENCODER
    c1 = conv_block(inputs, 64);  p1 = layers.MaxPool2D(2)(c1)
    c2 = conv_block(p1, 128);     p2 = layers.MaxPool2D(2)(c2)
    c3 = conv_block(p2, 256);     p3 = layers.MaxPool2D(2)(c3)
    c4 = conv_block(p3, 512);     p4 = layers.MaxPool2D(2)(c4)

    # BOTTLENECK
    c5 = conv_block(p4, 1024)

    # DECODER
    u6 = layers.Conv2DTranspose(512, 2, strides=2, padding='same')(c5)
    u6 = layers.concatenate([u6, c4]); c6 = conv_block(u6, 512)

    u7 = layers.Conv2DTranspose(256, 2, strides=2, padding='same')(c6)
    u7 = layers.concatenate([u7, c3]); c7 = conv_block(u7, 256)

    u8 = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(c7)
    u8 = layers.concatenate([u8, c2]); c8 = conv_block(u8, 128)

    u9 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(c8)
    u9 = layers.concatenate([u9, c1]); c9 = conv_block(u9, 64)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c9)
    return Model(inputs, outputs)

unet = build_unet()
unet.summary()

## 4. Training U-Net (Segmentasi)

In [ ]:
# ============================================================
# 4a. SPLIT DATA + COMPILE
# ============================================================
if imgs is not None and len(imgs) > 0:
    X_train, X_val, y_train, y_val = train_test_split(
        imgs, masks, test_size=0.2, random_state=SEED)
    print(f"Train: {len(X_train)}, Val: {len(X_val)}")

    unet.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=[iou_metric, dice_metric],
    )
    print("U-Net siap dilatih.")

In [ ]:
# ============================================================
# 4b. JALANKAN TRAINING SEGMENTASI
# ============================================================
# Hapus tanda komentar untuk melatih (butuh waktu & GPU).
# history_seg = unet.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     batch_size=BATCH_SIZE,
#     epochs=EPOCHS_SEG,
# )
# unet.save("unet_wbc_segmentation.h5")
# print("Model tersimpan: unet_wbc_segmentation.h5")

In [ ]:
# ============================================================
# 4c. PLOT KURVA TRAINING (jika sudah dilatih)
# ============================================================
# if 'history_seg' in dir():
#     plt.figure(figsize=(12,4))
#     plt.subplot(1,2,1)
#     plt.plot(history_seg.history['loss'], label='Train')
#     plt.plot(history_seg.history['val_loss'], label='Val')
#     plt.title('Loss U-Net'); plt.legend()
#     plt.subplot(1,2,2)
#     plt.plot(history_seg.history['dice_metric'], label='Train Dice')
#     plt.plot(history_seg.history['val_dice_metric'], label='Val Dice')
#     plt.title('Dice U-Net'); plt.legend()
#     plt.tight_layout(); plt.show()

## 5. Persiapan Data Klasifikasi

In [ ]:
# ============================================================
# 5. DATAGENERATOR KLASIFIKASI
# ============================================================
datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    validation_split=0.2,
)

if os.path.isdir(CLS_DATA_DIR):
    train_gen = datagen.flow_from_directory(
        CLS_DATA_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        subset="training", shuffle=True, seed=SEED,
    )
    val_gen = datagen.flow_from_directory(
        CLS_DATA_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        subset="validation", shuffle=False, seed=SEED,
    )
    print("\nIndeks kelas:", train_gen.class_indices)
else:
    train_gen = val_gen = None
    print("Dataset klasifikasi belum ada.")

## 6. Model Klasifikasi (ResNet50 Transfer Learning)

In [ ]:
# ============================================================
# 6. MODEL KLASIFIKASI
# ============================================================
def build_classifier(num_classes):
    base = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling="avg",
    )
    base.trainable = True  # fine-tuning
    x = layers.Dropout(0.3)(base.output)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=outputs)

if train_gen is not None:
    num_classes = len(train_gen.class_indices)
    classifier = build_classifier(num_classes)
    classifier.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    classifier.summary()
else:
    classifier = None
    print("Model klasifikasi tidak dibuat (dataset belum ada).")

## 7. Training Klasifikasi

In [ ]:
# ============================================================
# 7. JALANKAN TRAINING KLASIFIKASI
# ============================================================
# Hapus tanda komentar untuk melatih.
# if classifier is not None:
#     history_cls = classifier.fit(
#         train_gen,
#         validation_data=val_gen,
#         epochs=EPOCHS_CLS,
#     )
#     classifier.save("wbc_classifier.h5")
#     print("Model tersimpan: wbc_classifier.h5")

In [ ]:
# ============================================================
# 7b. PLOT KURVA KLASIFIKASI (jika sudah dilatih)
# ============================================================
# if 'history_cls' in dir():
#     plt.figure(figsize=(12,4))
#     plt.subplot(1,2,1)
#     plt.plot(history_cls.history['loss'], label='Train')
#     plt.plot(history_cls.history['val_loss'], label='Val')
#     plt.title('Loss Klasifikasi'); plt.legend()
#     plt.subplot(1,2,2)
#     plt.plot(history_cls.history['accuracy'], label='Train Acc')
#     plt.plot(history_cls.history['val_accuracy'], label='Val Acc')
#     plt.title('Akurasi Klasifikasi'); plt.legend()
#     plt.tight_layout(); plt.show()

## 8. Evaluasi & Confusion Matrix

In [ ]:
# ============================================================
# 8. EVALUASI KLASIFIKASI
# ============================================================
# if classifier is not None and not classifier.built:
#     classifier = tf.keras.models.load_model("wbc_classifier.h5")

# if classifier is not None:
#     val_gen.reset()
#     val_loss, val_acc = classifier.evaluate(val_gen)
#     print(f"Validasi Loss: {val_loss:.4f}, Akurasi: {val_acc:.4f}")
#
#     val_gen.reset()
#     y_pred = classifier.predict(val_gen)
#     y_pred_labels = np.argmax(y_pred, axis=1)
#     y_true_labels = val_gen.classes[:len(y_pred_labels)]
#
#     class_names = list(val_gen.class_indices.keys())
#     print("\n=== Classification Report ===")
#     print(classification_report(y_true_labels, y_pred_labels, target_names=class_names))
#
#     cm = confusion_matrix(y_true_labels, y_pred_labels)
#     disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
#     disp.plot(cmap="Blues")
#     plt.title("Confusion Matrix Klasifikasi WBC")
#     plt.show()

## 9. Prediksi pada Sampel Baru

In [ ]:
# ============================================================
# 9. PREDIKSI (Contoh)
# ============================================================
def predict_wbc(image_path, unet_model, clf_model, class_names):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    inp = np.expand_dims(img_resized / 255.0, axis=0)

    # Segmentasi
    mask = unet_model.predict(inp)[0, :, :, 0]
    mask = (mask > 0.5).astype(np.uint8)

    # Klasifikasi
    probs = clf_model.predict(inp)[0]
    cls = np.argmax(probs)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_resized); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(mask, cmap="gray"); axes[1].set_title("Segmented"); axes[1].axis("off")
    axes[2].barh(class_names, probs, color="steelblue")
    axes[2].set_title(f"Prediksi: {class_names[cls]}")
    plt.tight_layout(); plt.show()

    return class_names[cls], probs

# Contoh pemakaian (setelah kedua model dilatih):
# unet_model = tf.keras.models.load_model('unet_wbc_segmentation.h5', custom_objects={'iou_metric': iou_metric, 'dice_metric': dice_metric})
# clf_model = tf.keras.models.load_model('wbc_classifier.h5')
# predict_wbc('dataset/blood_cells/neutrophil/contoh.jpg', unet_model, clf_model, list(train_gen.class_indices.keys()))

## 9b. 📈 Grafik Hasil Training
Sel ini memuat riwayat training dari `hasil_training/riwayat_training.json` (dihasilkan oleh `train_full.py`) lalu menampilkan grafiknya secara langsung.

In [ ]:
# ============================================================
# 9b. GRAFIK HASIL TRAINING DARI FILE RIWAYAT
# ============================================================
import json
import os
import matplotlib.pyplot as plt

RIWAYAT = "hasil_training/riwayat_training.json"

if not os.path.exists(RIWAYAT):
    print("Riwayat belum ada. Jalankan train_full.py dulu:")
    print("  python train_full.py")
else:
    with open(RIWAYAT) as f:
        R = json.load(f)

    # Fallback: pakai riwayat klasifikasi v2 bila sudah ada
    R2 = "hasil_akhir/riwayat/riwayat_klasifikasi.json"
    if os.path.exists(R2):
        with open(R2) as f2:
            V = json.load(f2)
        if "klasifikasi_v2" in V:
            R["klasifikasi"] = V["klasifikasi_v2"]
            print("Menggunakan riwayat klasifikasi v2 (hasil_akhir/)")
        if "classification_report" in V:
            R["classification_report"] = V["classification_report"]

    # ---- GRAFIK SEGMENTASI (U-Net) ----
    if "segmentasi" in R:
        h = R["segmentasi"]
        ep = range(1, len(h["loss"]) + 1)
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
        axes[0].plot(ep, h["loss"], "b-o", label="Train Loss")
        axes[0].plot(ep, h["val_loss"], "r-o", label="Val Loss")
        axes[0].set_title("Segmentasi U-Net — Loss")
        axes[1].plot(ep, h["iou_metric"], "b-o", label="Train IoU")
        axes[1].plot(ep, h["val_iou_metric"], "r-o", label="Val IoU")
        axes[1].set_title("Segmentasi U-Net — IoU")
        axes[2].plot(ep, h["dice_metric"], "b-o", label="Train Dice")
        axes[2].plot(ep, h["val_dice_metric"], "r-o", label="Val Dice")
        axes[2].set_title("Segmentasi U-Net — Dice Coefficient")
        for a in axes:
            a.set_xlabel("Epoch"); a.legend(); a.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()
        print("Best:", R.get("segmentasi_best"))

    # ---- GRAFIK KLASIFIKASI (CNN) ----
    if "klasifikasi" in R:
        h = R["klasifikasi"]
        ep = range(1, len(h["loss"]) + 1)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        axes[0].plot(ep, h["loss"], "b-o", label="Train Loss")
        axes[0].plot(ep, h["val_loss"], "r-o", label="Val Loss")
        axes[0].set_title("Klasifikasi CNN — Loss")
        axes[1].plot(ep, h["accuracy"], "b-o", label="Train Akurasi")
        axes[1].plot(ep, h["val_accuracy"], "r-o", label="Val Akurasi")
        axes[1].set_title("Klasifikasi CNN — Akurasi")
        for a in axes:
            a.set_xlabel("Epoch"); a.legend(); a.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

    # ---- TABEL METRIK AKHIR ----
    if "classification_report" in R:
        rep = R["classification_report"]
        print("\n=== Metrik Akhir Klasifikasi ===")
        print(f"{'Kelas':<15} {'Precision':>9} {'Recall':>8} {'F1':>7} {'Support':>8}")
        for k, v in rep.items():
            if isinstance(v, dict) and "f1-score" in v:
                print(f"{k:<15} {v['precision']:>9.4f} {v['recall']:>8.4f} {v['f1-score']:>7.4f} {int(v['support']):>8}")
        if "accuracy" in rep:
            print(f"\nAkurasi keseluruhan: {rep['accuracy']:.4f}")

## 9c. Gambar Hasil Training (langsung dari folder `hasil_training/`)

In [ ]:
# ============================================================
# 9c. TAMPILKAN GAMBAR GRAFIK DARI hasil_training/
# ============================================================
from IPython.display import Image, display
import glob

pngs = sorted(glob.glob("hasil_training/*.png"))
if pngs:
    for p in pngs:
        print(p)
        display(Image(filename=p, width=640))
else:
    print("Belum ada grafik. Jalankan train_full.py terlebih dahulu:")
    print("  python train_full.py")

## 10. Simpan & Unduh Model

In [ ]:
# ============================================================
# 10. MENYIMPAN MODEL DALAM FORMAT .h5 & .keras
# ============================================================
# converter = tf.keras.models.load_model('unet_wbc_segmentation.h5',
#                                      custom_objects={'iou_metric': iou_metric, 'dice_metric': dice_metric})
# converter.save('unet_wbc_segmentation.keras')

# classifier2 = tf.keras.models.load_model('wbc_classifier.h5')
# classifier2.save('wbc_classifier.keras')

print("Selesai. Model siap disimpan/diunduh.")

## 📚 Referensi Jurnal (dipendekkan)
- Benaissa I, et al. (2025). *Gated Attention-Augmented Double U-Net for WBC Segmentation.* J Imaging, 11(11):386.
- Zhu M, et al. (2023). *Improved U-Net-based leukocyte segmentation method.* J Biomed Opt, 28(4):045002.
- Cheuque C, et al. (2022). *An Efficient Multi-Level CNN Approach for WBC Classification.* Diagnostics, 12(2):248.
- Islam O, et al. (2024). *An explainable AI-based blood cell classification using optimized CNN.* J Pathol Inform, 15:100389.
- Ronneberger O, et al. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation.* MICCAI.
- He K, et al. (2016). *Deep Residual Learning for Image Recognition.* CVPR.

> Lihat `DATASET_DAN_REFERENSI.md` untuk daftar lengkap (16 referensi + detail dataset).